# TP Jour 1 — Choix des Datasets Kaggle (Mono ou Multi-Sources), HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et de combiner **un dataset multi-tables ou plusieurs datasets publics complémentaires sur Kaggle** afin de les croiser par jointure.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner **un ou plusieurs datasets Kaggle** (~50 à 500 Mo au total) partageant au moins un axe de jointure (date/heure, zone géographique, code ISO/catégorie, identifiant métier).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts multi-sources sur HDFS** (`/lakehouse/bronze/raw/<source>/...`) et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** avec traçabilité de la source (`_source_dataset`, `_source_file`, `_ingested_at`) et inspecter vos jobs sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de votre projet *(À compléter par le groupe)*
- **Titre du sujet :** `Le prix des courses VTC à Boston : Uber vs Lyft face à l'heure, au trajet et à la météo (26 nov. – 18 déc. 2018)`
- **Dataset(s) Kaggle retenu(s) (`auteur/nom-du-dataset`) :**
  1. Source principale : `ravi72munde/uber-lyft-cab-prices` → table **`cab_rides.csv`** (1 ligne = 1 course Uber ou Lyft : prix, distance, trajet, offre, majoration)
  2. Source(s) complémentaire(s) pour jointure : même dataset multi-tables → table **`weather.csv`** (relevés météo horaires des 12 quartiers : température, pluie, nuages, humidité, vent, pression)
- **Clé(s) de jointure envisagée(s) entre sources / tables :** `quartier + heure` : `cab_rides.source` = `weather.location` (noms normalisés) **et** horodatages ramenés à la même heure UTC (millisecondes côté courses, secondes côté météo). Jointure secondaire par nomenclature métier `product_id` → gamme du véhicule. *(ex. date + ville, code aéroport, code postal, catégorie...)*
- **Problématique métier principale :** `Qu'est-ce qui fait varier le prix d'une course VTC à Boston : l'heure, le trajet, la distance, la compagnie (Uber / Lyft), la gamme de véhicule, la majoration « surge » ou la météo (pluie, froid) ?`
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :**
  - *Temporelles* : `time_stamp` des courses (epoch ms) et de la météo (epoch s) → heure locale de Boston, jour, tranche horaire
  - *Quantitatives* : `price`, `distance`, `surge_multiplier` (courses) ; `temp`, `rain`, `clouds`, `humidity`, `wind`, `pressure` (météo)
  - *Catégorielles / géographiques* : `cab_type` (Uber / Lyft), `name` (offre : Shared, UberX, Lux…), `source` / `destination` / `location` (12 quartiers de Boston)

## 1. Téléchargement de vos Datasets Kaggle dans `/workspace/data_raw/<source>/`

Renseignez dans la liste `KAGGLE_DATASETS` un ou plusieurs identifiants Kaggle (ex. `["auteur1/dataset-principal", "auteur2/meteo-ou-referentiel"]`), ou déposez directement vos fichiers/sous-dossiers dans `data_raw/`.
- Chaque dataset est isolé dans son propre sous-dossier `data_raw/<nom-dataset>/` pour éviter toute collision de noms (`train.csv`, `metadata.csv`, etc.) entre plusieurs sources.
- **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [1]:
import re
import shutil
from pathlib import Path

DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# TODO : Renseignez un ou plusieurs datasets Kaggle à combiner (format "auteur/nom-du-dataset")
KAGGLE_DATASETS: list[str] = [
    "ravi72munde/uber-lyft-cab-prices",  # dataset multi-tables : cab_rides.csv (faits) + weather.csv (jointure)
]

for slug in KAGGLE_DATASETS:
    slug = slug.strip()
    if not slug:
        continue
    subdir = re.sub(r"[^a-zA-Z0-9._-]+", "_", slug.split("/")[-1]).strip("_") or "dataset"
    target_dir = DATA_RAW / subdir
    existing = [f for f in target_dir.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS] if target_dir.exists() else []
    if not existing:
        import kagglehub
        print(f"Téléchargement de {slug} -> {target_dir}/ ...")
        cache_dir = Path(kagglehub.dataset_download(slug))
        target_dir.mkdir(parents=True, exist_ok=True)
        for src_file in sorted(cache_dir.rglob("*")):
            if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
                dest_file = target_dir / src_file.relative_to(cache_dir)
                dest_file.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(src_file, dest_file)

local_files = sorted([f for f in DATA_RAW.rglob("*") if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, (
    "Aucun fichier détecté dans /workspace/data_raw/. "
    "Renseignez `KAGGLE_DATASETS` ci-dessus ou utilisez `task data:download DATASET='auteur1/ds1,auteur2/ds2'`."
)

print(f"Inventaire local multi-sources dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    rel = f.relative_to(DATA_RAW)
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {str(rel):<55} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

Inventaire local multi-sources dans /workspace/data_raw (2 fichier(s)) :
 - uber-lyft-cab-prices/cab_rides.csv                        84.65 Mo  (~6 bloc(s) HDFS de 16 Mo)
 - uber-lyft-cab-prices/weather.csv                           0.33 Mo  (~1 bloc(s) HDFS de 16 Mo)


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# La config HDFS + Delta Lake est chargée automatiquement depuis config/spark-defaults.conf
spark = SparkSession.builder.appName("J1_HDFS_Ingestion_Bronze").getOrCreate()

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# Les 3 paramètres HDFS lus depuis la configuration de la session
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

Spark version       : 4.2.0
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [3]:
jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()
fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")
fs.mkdirs(raw_hdfs_dir)

for file_path in local_files:
    rel_posix = file_path.relative_to(DATA_RAW).as_posix()
    src = PathHDFS(f"file://{file_path}")
    dst = PathHDFS(f"/lakehouse/bronze/raw/{rel_posix}")
    fs.mkdirs(dst.getParent())
    fs.copyFromLocalFile(False, True, src, dst)
    print(f"Téléversé vers HDFS : {dst}")

Téléversé vers HDFS : /lakehouse/bronze/raw/uber-lyft-cab-prices/cab_rides.csv
Téléversé vers HDFS : /lakehouse/bronze/raw/uber-lyft-cab-prices/weather.csv


In [4]:
# Inspection récursive du découpage en blocs de 16 Mo sur l'ensemble des sources ingérées
remote_iter = fs.listFiles(raw_hdfs_dir, True)
statuses = []
while remote_iter.hasNext():
    statuses.append(remote_iter.next())
statuses.sort(key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    rel_hdfs = path.toString().split("/lakehouse/bronze/raw/", 1)[-1]
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {rel_hdfs} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : uber-lyft-cab-prices/cab_rides.csv (84.65 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #2 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #3 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #4 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #5 ( 4.65 Mo) -> DataNodes : ['datanode1', 'datanode2']
Fichier : uber-lyft-cab-prices/weather.csv (0.33 Mo | repl=2)
   Bloc #0 ( 0.33 Mo) -> DataNodes : ['datanode3', 'datanode2']


### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.
2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?
3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

### Réponses aux questions d'analyse

**1. Nombre de blocs du plus gros fichier.**
`cab_rides.csv` pèse **84,65 Mo**. 84,65 / 16 = 5,3, donc arrondi au supérieur **6 blocs** : 5 blocs pleins de 16 Mo + 1 dernier bloc de **4,65 Mo**. `weather.csv` (0,33 Mo) tient dans **1 seul bloc**. On le vérifie dans la cellule ci-dessus, avec `hdfs fsck` (« Total blocks : 7 ») et dans `Utilities > Browse the file system` sur http://localhost:9870.

**2. Un fichier de 3 Mo bloque-t-il 16 Mo de disque ?**
Non. 16 Mo est une taille **maximale** de bloc, pas une réservation : chaque bloc est stocké comme un fichier ordinaire sur le disque du DataNode, à sa taille réelle. `weather.csv` occupe 0,33 Mo par copie, et le dernier bloc de `cab_rides.csv` 4,65 Mo, pas 16. Le vrai coût d'un petit fichier est ailleurs : chaque fichier et chaque bloc prend de la place dans la **mémoire du NameNode** (environ 150 octets par objet). Des millions de petits fichiers saturent donc le NameNode : c'est le « small files problem ».

**3. Pourquoi `dfs.replication = 2` sur 3 DataNodes ?**
- Chaque bloc existe en 2 exemplaires sur 2 machines différentes : on peut **perdre une machine sans perdre de donnée**.
- Il reste toujours **une 3e machine libre** pour recréer la copie manquante après une panne (voir le crash-test). Avec une réplication de 3 sur 3 DataNodes, chaque machine aurait déjà tout : après une panne, impossible de revenir à 3 copies, et les blocs resteraient sous-répliqués.
- On consomme 2 fois l'espace disque au lieu de 3.

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [5]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# Lecture du CSV pendant que datanode2 est arrêté : HDFS sert les blocs depuis les copies survivantes
df_check = spark.read.option("header", "true").csv(largest_hdfs_path)

assert df_check is not None, "TODO : Lisez `largest_hdfs_path` dans `df_check`."

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

# Par défaut, Spark découpe le fichier pour occuper ses cœurs (taille d'un morceau = taille totale / nb de cœurs,
# plancher 4 Mo) : le nombre de partitions dépend donc du nombre de cœurs, pas des blocs HDFS.
print(f"Cœurs Spark (defaultParallelism) : {spark.sparkContext.defaultParallelism}")

# Bonus : « 1 bloc HDFS = 1 partition Spark » en fixant la taille max d'un morceau à 16 Mo (= 1 bloc)
# et en levant le découpage par cœur (minPartitionNum = 1)
spark.conf.set("spark.sql.files.maxPartitionBytes", block_size)
spark.conf.set("spark.sql.files.minPartitionNum", "1")
df_blocks = spark.read.option("header", "true").csv(largest_hdfs_path)
print(f"Partitions avec 1 morceau = 1 bloc de 16 Mo : {df_blocks.rdd.getNumPartitions()}")
spark.conf.unset("spark.sql.files.maxPartitionBytes")
spark.conf.unset("spark.sql.files.minPartitionNum")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/uber-lyft-cab-prices/cab_rides.csv


Nombre total de lignes lues       : 693,071
Nombre de partitions Spark créées : 22
Cœurs Spark (defaultParallelism) : 22


Partitions avec 1 morceau = 1 bloc de 16 Mo : 6


### Observations du crash-test et réponse à l'exercice 4.1

| Étape | Mesure |
|---|---|
| État initial (`task hdfs:fsck`) | 7 blocs, réplication moyenne 2.0, statut **HEALTHY** ; **2 blocs** avaient une copie sur `datanode2` (le bloc #5 de `cab_rides.csv` et le bloc de `weather.csv`) |
| `task hdfs:crash-dn2` | `datanode2` déclaré **Dead** sur http://localhost:9870 après **47 à 51 s** selon les essais (théorie : 2 × 10 s + 10 × 3 s = 50 s) |
| Pendant la panne | **0 bloc manquant**, statut toujours HEALTHY, mais **2 blocs sous-répliqués** (réplication moyenne 1,71 : une seule copie vivante) |
| Auto-réparation | ~14 s après la détection, le NameNode repère les 2 blocs et ordonne leur recopie vers le 3e DataNode ; **réplication revenue à 2.0 à 93 s**, `datanode2` toujours éteint |
| `task hdfs:restore-dn2` | `datanode2` revenu dans le cluster en ~4 s |

*Le chronométrage de la réparation a été refait lors d'un 2e crash-test : au premier, la mesure s'était arrêtée trop tôt (juste après la détection, le NameNode n'avait pas encore recompté les blocs).*

**Continuité de service :** pendant que `datanode2` était éteint, Spark a lu **693 071 lignes** sans aucune erreur : pour chaque bloc, le client HDFS va chercher la copie survivante.

**Lien partitions / blocs :**
- Par défaut, Spark a créé **22 partitions** pour **6 blocs**. Spark ne découpe pas un fichier selon les blocs HDFS, mais selon `spark.sql.files.maxPartitionBytes` (128 Mo par défaut) et le **nombre de cœurs** (22 ici). Pour occuper tous ses cœurs, il coupe le fichier en morceaux d'environ 4 Mo (taille du fichier ÷ 22 cœurs, avec un plancher de 4 Mo).
- En fixant la taille d'un morceau à 16 Mo (`maxPartitionBytes`) et en levant ce découpage par cœur (`minPartitionNum = 1`), on obtient **6 partitions = 6 blocs** : une tâche Spark par bloc.
- Sur un vrai cluster, chaque tâche s'exécute alors sur la machine qui stocke son bloc (**Data Locality** : on déplace le calcul vers la donnée). Ici, Spark tourne en `local[*]` dans le conteneur `spark-jupyter`, donc la localité n'est pas exploitée physiquement, mais le découpage est le même.

**Couche Bronze :** 2 tables Delta, `cab_rides` (693 071 lignes) et `weather` (6 276 lignes), avec leur lignage (`_source_dataset`, `_source_file`, `_ingested_at`). Au passage, l'en-tête de `weather.csv` commence par un caractère invisible (BOM UTF-8) qui aurait renommé la 1re colonne en `\ufefftemp` : il est retiré lors de la normalisation des noms de colonnes.

---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts (mono ou multi-datasets) en tables Delta Lake Bronze
Dans une architecture **Médaillon multi-sources**, la couche **Bronze** conserve chaque table brute fidèle à sa source tout en l'enrichissant de métadonnées de lignage (*lineage*) indispensables avant les jointures du Jour 2 :
- `_ingested_at` : `F.current_timestamp()`
- `_source_dataset` : `F.lit(dataset_name)` *(identifie le dataset d'origine)*
- `_source_file` : `F.col("_metadata.file_path")`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de vos datasets dans `/lakehouse/bronze/<nom_table>`.

In [6]:
import re

# TODO : Définissez vos tables Bronze sous la forme :
#   "nom_table_bronze": ("nom_dataset_source", "/lakehouse/bronze/raw/<sous-dossier>/<fichier>")
BRONZE_SOURCES: dict[str, tuple[str, str]] = {
    "cab_rides": ("uber-lyft-cab-prices", "/lakehouse/bronze/raw/uber-lyft-cab-prices/cab_rides.csv"),  # faits
    "weather":   ("uber-lyft-cab-prices", "/lakehouse/bronze/raw/uber-lyft-cab-prices/weather.csv"),    # jointure
}

assert len(BRONZE_SOURCES) > 0, "TODO : Renseignez vos tables à ingérer dans `BRONZE_SOURCES` (idéalement >= 2 tables à joindre au Jour 2)."

for table_name, source_spec in BRONZE_SOURCES.items():
    dataset_name, raw_path = source_spec if isinstance(source_spec, tuple) else ("default", source_spec)
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    # 1. Lecture du CSV brut (en-tête + typage automatique) + chemin du fichier source (lignage)
    df_raw = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(raw_path)
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    assert df_raw is not None, f"TODO : Construisez et écrivez la table Bronze `{table_name}`."

    # 2. Noms de colonnes en snake_case, sans espaces ni points, et sans le BOM UTF-8 invisible (\ufeff)
    #    présent au début de weather.csv (sinon la 1re colonne s'appellerait « \ufefftemp » au lieu de « temp »)
    df_raw = df_raw.toDF(*[
        c if c.startswith("_") else
        re.sub(r"(?<=[a-z0-9])(?=[A-Z])", "_", c.replace("\ufeff", "").strip()).lower().replace(" ", "_").replace(".", "_")
        for c in df_raw.columns
    ])

    # 3. Métadonnées de lignage
    df_raw = (
        df_raw
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_dataset", F.lit(dataset_name))
    )

    # 4. Écriture au format Delta Lake sur HDFS
    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )

    cnt = spark.read.format("delta").load(bronze_delta_path).count()
    print(f"[BRONZE OK] {bronze_delta_path:<38} (source={dataset_name:<18}) -> {cnt:>10,} lignes")

[BRONZE OK] /lakehouse/bronze/cab_rides            (source=uber-lyft-cab-prices) ->    693,071 lignes


[BRONZE OK] /lakehouse/bronze/weather              (source=uber-lyft-cab-prices) ->      6,276 lignes


In [7]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['cab_rides', 'weather']
